In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from model_v2 import NSConfig

In [ ]:
## inference with KV-Cache


# 1. Rope
def init_rope(cfg: NSConfig):

    inv_freq = 1.0 / (cfg.rope_theta ** (torch.arange(0, cfg.head_dim, 2, dtype=torch.float32) / cfg.head_dim))

    pos = torch.arange(0, cfg.context_length, dtype=torch.float32)

    angles = torch.einsum("i,j -> ij", pos, inv_freq)

    angles = torch.cat([angles, angles], dim=-1)

    cos = angles.cos()
    sin = angles.sin()

    return cos, sin


def rotate_half(x):

    head_dim = x.shape[-1]
    assert head_dim % 2 == 0, "head dim must be even"
    x1, x2 = x[:, :, :, : head_dim // 2], x[:, :, :, head_dim // 2 :]
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(q, k, cos, sin, position_offset=0):

    seq_len = q.shape[-2]
    cos = cos[position_offset : position_offset + seq_len].unsqueeze(0).unsqueeze(0)
    sin = sin[position_offset : position_offset + seq_len].unsqueeze(0).unsqueeze(0)

    q_rotated = q * cos + rotate_half(q) * sin
    k_rotated = k * cos + rotate_half(k) * sin

    return q_rotated.to(dtype=q.dtype), k_rotated.to(dtype=k.dtype)


# 2. GQA Attention
class GQAAttention(nn.Module):

    def __init__(self, cfg: NSConfig):
        super().__init__()

        assert cfg.n_heads % cfg.n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"

        self.cfg = cfg
        self.kv_group_size = cfg.n_heads // cfg.n_kv_heads

        self.w_q = nn.Linear(cfg.emb_dim, cfg.n_heads * cfg.head_dim, bias=False)
        self.w_k = nn.Linear(cfg.emb_dim, cfg.n_kv_heads * cfg.head_dim, bias=False)
        self.w_v = nn.Linear(cfg.emb_dim, cfg.n_kv_heads * cfg.head_dim, bias=False)
        self.w_o = nn.Linear(cfg.n_heads * cfg.head_dim, cfg.emb_dim, bias=False)


    def forward(self, x, cos, sin, past_key_value=None, use_cache=False):
        """
            x: [B, L, D]
            cos: [L, D]
            sin: [L, D]
            past_key_value: (k, v)  past_key_value是一个元组，元组的第一个元素是k，第二个元素是v
            v的形状是[B, L, H, D]，表示历史token的V值，K同理

            use_cache: bool
        """

        bsz, new_tokens, _ = x.shape

        q = self.w_q(x).view(bsz, new_tokens, self.cfg.n_heads, self.cfg.head_dim).transpose(1, 2)
        k = self.w_k(x).view(bsz, new_tokens, self.cfg.n_kv_heads, self.cfg.head_dim).transpose(1, 2)
        v = self.w_v(x).view(bsz, new_tokens, self.cfg.n_kv_heads, self.cfg.head_dim).transpose(1, 2)

        if past_key_value is None:
            past_k = None
            past_v = None
            past_len = 0
        else:
            past_k, past_v = past_key_value
            past_len = past_k.shape[-2]

        q, k = apply_rope(q, k, cos, sin, position_offset=past_len)

        if past_k is not None:
            k = torch.cat([past_k, k], dim=-2)
            v = torch.cat([past_v, v], dim=-2)

        present_key_value = (k, v)


        k_attn = torch.repeat_interleave(k, self.kv_group_size, dim=1)
        v_attn = torch.repeat_interleave(v, self.kv_group_size, dim=1)

        total_len = k_attn.shape[-2]

        if past_len == 0:
            # 标准 causal attention
            attn_output = F.scaled_dot_product_attention(
                q,
                k_attn,
                v_attn,
                attn_mask=None,
                dropout_p=0.0,
                is_causal=True,
            )  # [B, H, L, D]

        elif new_tokens == 1:
            # Decode: Q=1 token, K/V = past_tokens + 1 token
            attn_output = F.scaled_dot_product_attention(
                q,
                k_attn,
                v_attn,
                attn_mask=None,
                dropout_p=0.0,
                is_causal=False,
            )
        else:

            q_positions = torch.arange(past_len, past_len + new_tokens, device=x.device)
            k_positions = torch.arange(0, total_len, device=x.device)


            # new_tokens 表示未来的token
            future_mask = q_positions.unsqueeze(1) < k_positions.unsqueeze(0)

            # q_postions = 
            #    Q5    5    5   5   5   5   5   5
            #    Q6    6    6   6   6   6   6   6
            #    Q7    7    7   7   7   7   7   7

            # k_postions = 
            #         K0   K1  K2  K3  K4  K5  K6  K7
            #          0    1   2   3   4   5   6   7
            #          0    1   2   3   4   5   6   7
            #          0    1   2   3   4   5   6   7
            #         ||
            #         ||
            #         \/
            
            #          K0   K1  K2  K3  K4  K5  K6  K7
            #    Q5    0    0   0   0   0   0   1   1
            #    Q6    0    0   0   0   0   0   0   1
            #    Q7    0    0   0   0   0   0   0   0

            attn_mask = torch.zeros((new_tokens, total_len), dtype=q.dtype, device=x.device)

            attn_mask = attn_mask.masked_fill(future_mask, -torch.inf)

            attn_output = F.scaled_dot_product_attention(
                q,
                k_attn,
                v_attn,
                attn_mask=attn_mask,
                dropout_p=0.0,
                is_causal=False,
            )


        context_v = attn_output.transpose(1, 2).contiguous().view(bsz, new_tokens, self.cfg.n_heads * self.cfg.head_dim)

        output = self.w_o(context_v)

        if use_cache:
            return output, present_key_value

        return output


# 3. RMS Norm
class RMSNorm(nn.Module):

    def __init__(self, hidden_dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(hidden_dim))


    def forward(self, x):
        variance = x.float().pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.eps).to(x.dtype)
        return x * self.weight


# 4. FFN
class FeedForward(nn.Module):

    def __init__(self, cfg: NSConfig):
        super().__init__()

        self.fc1 = nn.Linear(cfg.emb_dim, cfg.hidden_dim, bias=False)
        self.fc2 = nn.Linear(cfg.emb_dim, cfg.hidden_dim, bias=False)
        self.fc3 = nn.Linear(cfg.hidden_dim, cfg.emb_dim, bias=False)

    def forward(self, x):
        x1 = self.fc1(x)
        x2 = self.fc2(x)
        x = F.silu(x1) * x2

        return self.fc3(x)


# 5. Transformer Block
class TransformerBlock(nn.Module):

    def __init__(self, cfg: NSConfig):

        super().__init__()

        self.cfg = cfg

        self.rms_norm1 = RMSNorm(cfg.emb_dim, cfg.rms_norm_eps)
        self.attn = GQAAttention(cfg)

        self.rms_norm2 = RMSNorm(cfg.emb_dim, cfg.rms_norm_eps)
        self.ffn = FeedForward(cfg)

    

    def forward(self, x, cos, sin, past_key_value=None, use_cache=False):

        residual = x
        x = self.rms_norm1(x)

        if use_cache:
            x, present_key_value = self.attn(x, cos, sin, past_key_value=past_key_value, use_cache=True)
        else:
            x = self.attn(x, cos, sin, past_key_value=None, use_cache=False)

        x = residual + x 

        residual = x
        x = self.rms_norm2(x)
        x = self.ffn(x)
        x = residual + x

        if use_cache:
            return x, present_key_value

        return x


# 6. NsModel
class NsModelWithKVCache(nn.Module):

    def __init__(self, cfg: NSConfig):

        super().__init__()

        self.cfg = cfg

        self.emb = nn.Embedding(cfg.vocab_size, cfg.emb_dim)

        cos, sin = init_rope(cfg)

        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

        self.layers = nn.ModuleList([
            TransformerBlock(cfg) for _ in range(cfg.n_layers)
        ])

        self.final_norm = RMSNorm(cfg.emb_dim, cfg.rms_norm_eps)
        self.lm_out = nn.Linear(cfg.emb_dim, cfg.vocab_size, bias=False)

    def forward(self, x, past_key_values=None, use_cache=False):
        """
        past_key_values/present_key_values 结构
        [
            "layer_0": (present_k, present_v),
            "layer_1": (present_k, present_v),
            ...
            "layer_n": (present_k, present_v),
        ]
        """

        bsz, seq_len = x.shape

        if past_key_values is None:
            past_len = 0
        else:
            past_len = past_key_values[0][0].shape[-2]

        if past_len + seq_len > self.cfg.context_length:
            raise ValueError(f"Input sequence length {seq_len} + past sequence length {past_len} exceeds max length {self.cfg.context_length}")


        x = self.emb(x)


        # 每层分别保存 KV cache
        if use_cache:
            if past_key_values is None:
                past_key_values = [None] * self.cfg.n_layers

            # >>> a = [(1, 2), (3,4)]
            # >>> b = a
            # >>> a = []  # NOTE: present_key_values每次decode时置为[]无影响，因为每次decode完成后，present_key_values已经赋值给past_key_values了
            # >>> b
            # [(1, 2), (3, 4)]

            present_key_values = []

        for layer_idx, layer in enumerate(self.layers):

            if use_cache:
                x, present_kv = layer(x, self.cos, self.sin, past_key_value=past_key_values[layer_idx], use_cache=True)
                present_key_values.append(present_kv)
            else:
                x = layer(x, self.cos, self.sin)

        x = self.final_norm(x)

        logits = self.lm_out(x)

        if use_cache:

            return logits, present_key_values

        return logits


In [18]:
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file("/home/sllm_scratch/D2L/nsllm/model/tokenizer_fixed.json")

In [19]:
cfg = NSConfig()

device = "cuda:1"

model = NsModelWithKVCache(cfg)
model.to(device)

model.load_state_dict(
    torch.load("/home/sllm_scratch/D2L/nsllm/model/ns_model_v2_epoch_1_step_10w.bin", map_location=device, weights_only=True)
)
model.to(device)
model.eval()


NsModelWithKVCache(
  (emb): Embedding(32000, 768)
  (layers): ModuleList(
    (0-11): 12 x TransformerBlock(
      (rms_norm1): RMSNorm()
      (attn): GQAAttention(
        (w_q): Linear(in_features=768, out_features=768, bias=False)
        (w_k): Linear(in_features=768, out_features=128, bias=False)
        (w_v): Linear(in_features=768, out_features=128, bias=False)
        (w_o): Linear(in_features=768, out_features=768, bias=False)
      )
      (rms_norm2): RMSNorm()
      (ffn): FeedForward(
        (fc1): Linear(in_features=768, out_features=2048, bias=False)
        (fc2): Linear(in_features=768, out_features=2048, bias=False)
        (fc3): Linear(in_features=2048, out_features=768, bias=False)
      )
    )
  )
  (final_norm): RMSNorm()
  (lm_out): Linear(in_features=768, out_features=32000, bias=False)
)

In [20]:
prompt = "fuck you"

input_ids = tokenizer.encode(prompt).ids

eos_token_id = tokenizer.token_to_id("<eos>")
input_ids = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)

In [8]:
max_output = 1500

with torch.no_grad():

    
    for _ in range(max_output):

        logits = model.forward(input_ids)

        logits = logits[:, -1, :]

        # 贪心
        # next_token = torch.argmax(logits, dim=-1, keepdim=True)

        # 概率
        probs = torch.softmax(
            logits,
            dim=-1
        )
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        if next_token.item() == eos_token_id:
            print("-------------end-----------------------")
            break
        input_ids = torch.cat([input_ids, next_token], dim=1)




output = tokenizer.decode(input_ids.squeeze(0).cpu().numpy())
print(output)


 fuck you at a drop noted museum that includes chambers that appear at every right hand of the museum.
Just as they hide their stars in the spiral in the floor-below striking awe. A quasi-karate theatre is tucked between desk and screen. It is an impressive feature, but even with the exception of the Big Count’s finest number of retirement services, which we’ve only seen in sector years, nobody actually says the off-air job is cheap and there’s no real way to handle the Guitar Theatre.
Yes, there are some things with a computer-like design and an aptitude for editing and photographing skills. In the good offices and feature spots, one of them even captures the instruments, made from computer ’56’s Paul Nash-type components.
Don’t get me wrong, there are a number of unadorned features, including the huge heads placed over every faceted surface, for example J Capitol Art Base, a well-equipped domicile for Mia Mitchell Stryker.
There is some cute bit at the back, though. The enormous proj

In [15]:
prompt = "fuck you"

input_ids = tokenizer.encode(prompt).ids

eos_token_id = tokenizer.token_to_id("<eos>")
input_ids = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0).to(device)

In [21]:
max_output = 1500

with torch.no_grad():


    # 1. prefill
    logits, past_key_values = model.forward(input_ids, use_cache=True)

    generated = input_ids


    for _ in range(max_output):

        logits = logits[:, -1, :]


        # 概率
        probs = torch.softmax(
            logits,
            dim=-1
        )
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        generated = torch.cat([generated, next_token], dim=1)

        if next_token.item() == eos_token_id:
            print("-------------end-----------------------")
            break

        logits, past_key_values = model(next_token, past_key_values=past_key_values, use_cache=True)

    

output = tokenizer.decode(generated.squeeze(0).cpu().numpy())
print(output)


-------------end-----------------------
 fuck you’ played straight you wiped the www.mypoloso boss along with the low price of a vulva on this great Canon Gold appropriately tipped off Claptor pull top, but failing nevertheless, we don’t think it would be worth a cruise between.- W4 1900
Into a dry sea-bathermoon in Mykonos offers a la Crocca at sea. shin reefs dot the coast of the west-south coast, the bulbasic world’s most stony coastal mammal, waters rising towards Daecapeau, the Straits of San Sushi, Kathmandu wild rice making Judith massacre as Buddha’s pup and Yodir is in need of a centre wide diversion before there are any Belt hops to meet. The octopus is separated nauru in a drysuit dot as it supposedly joins a maturing swan above or dispersed by the Convergi constellation flowing into the nightThuan und Jongham live-in on fowl
- Same diagonal all-inclusive cliff in Edinburgh, Scotland
- Travelsized yet beautiful with Bertelli architecture and on location
- Sleeping in Almair 